# Ollama + Llava на Google Colab
Запускаем Ollama с моделью llava и открываем доступ через туннель.

In [ ]:
!apt-get update -qq
!apt-get install -y -qq wget curl cmake build-essential

# Устанавливаем Ollama
!curl -fsSL https://ollama.com/install.sh | sh

import os
os.environ['OLLAMA_HOST'] = '0.0.0.0:11434'
os.environ['OLLAMA_KEEP_ALIVE'] = '24h'

# Запускаем сервер в фоне
get_ipython().system_raw('ollama serve &')
import time
time.sleep(5)
print('Ollama запущен')

In [ ]:
# Качаем модель llava (4.1 GB)
!ollama pull llava
print('Модель готова')

In [ ]:
# Устанавливаем cloudflared (туннель)
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O cloudflared
!chmod +x cloudflared

# Запускаем туннель к Ollama
import subprocess
import threading

def run_tunnel():
    proc = subprocess.Popen(
        ['./cloudflared', 'tunnel', '--url', 'http://localhost:11434'],
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True
    )
    for line in proc.stdout:
        print(line, end='')
        if 'trycloudflare.com' in line:
            # Парсим URL
            import re
            match = re.search(r'https://[a-z0-9-]+\.trycloudflare\.com', line)
            if match:
                print(f'\n\n=== ТВОЙ URL (вставь в код бота) ===')
                print(f'{match.group()}/api/chat')
                print(f'===================================\n')

thread = threading.Thread(target=run_tunnel, daemon=True)
thread.start()

# Ждём URL
print('Ожидаю туннель... (до 30 сек)')
time.sleep(30)
print('Туннель запущен. Держи Colab открытым.')

## Что делать:
1. Скопируй URL из вывода выше (вида `https://что-то.trycloudflare.com/api/chat`)
2. Открой файл бота `bot/services/openrouter.py` 
3. Замени `http://localhost:11434/api/chat` на этот URL
4. Делай это при каждом перезапуске Colab (URL меняется)